# Notebook 3: Primeiros Passos com LangChain

Aqui você vai conhecer o **LangChain**, um dos frameworks mais populares para construção de aplicações com Modelos de Linguagem de Grande Escala (LLMs).

Ao longo deste notebook você aprenderá a:

- Instalar e configurar o ambiente LangChain com o provedor Groq.
- Criar **chains** simples usando a sintaxe LCEL (LangChain Expression Language).
- Construir **pipelines** com etapas de pré e pós-processamento.
- Definir **tools** customizadas que o LLM pode invocar.
- Implementar **roteamento dinâmico** de perguntas com `RunnableBranch`.
- Adicionar **resiliência** ao seu pipeline com retry e fallback.

> **Pré-requisitos:** conhecimento básico de Python e ter concluído os Notebooks 1 e 2 desta série (Para obter a API do Groq).

---
## Seção 1: Configuração do Ambiente

Antes de escrever qualquer linha de código com LangChain, precisamos instalar as bibliotecas necessárias e configurar as credenciais de acesso ao provedor de LLM.

### O que é o Groq?

**Groq** é uma plataforma de inferência de LLMs com altíssima velocidade e com um nível gratuito generoso. Neste notebook usamos Groq como provedor de LLM em vez da OpenAI diretamente. A integração é feita pelo pacote `langchain-groq`.

### O que é uma API Key?

Uma **API Key** é uma senha que identifica a sua conta junto ao serviço. Ela deve ser mantida em segredo. No Google Colab, armazenamos segredos pela aba lateral **🔑 Secrets** (ícone de chave) para não expor a key no código.

In [ ]:
# ============================================================================
# 1. SETUP E INSTALAÇÃO
# ============================================================================

# Instalar dependências
!pip install -q langchain-groq langchain-core langchain-community python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


---
### Milestone 1 Ambiente Pronto

**O que se espera?**  
A célula abaixo de configuração do Groq deve imprimir `"Groq configurado com sucesso!"`.

**Sua tarefa:**  
- Execute a célula de instalação e confirme que não há erros.
- Acesse a aba **🔑 Secrets** do Colab, crie um segredo chamado `GROQ_API_KEY` e cole sua chave da API Groq. Se ainda não tem uma conta, crie em [groq.com](https://groq.com) gratuitamente.
- Execute a célula de configuração e verifique se a mensagem de sucesso aparece.

---

In [ ]:
# ============================================================================
# 2. CONFIGURAÇÃO DO GROQ
# ============================================================================

from google.colab import userdata  # Módulo do Colab para acessar segredos
import os

# Obter API key do Groq (armazenada em Secrets do Colab)
groq_api_key = userdata.get('GROQ_API_KEY')  # Lê o segredo pelo nome exato

# Exportar a key como variável de ambiente, que o LangChain lê automaticamente
os.environ['GROQ_API_KEY'] = groq_api_key

print("Groq configurado com sucesso!")

Groq configurado com sucesso!


---
## Seção 2: LCEL e a Chain Simples

### O que é LCEL?

**LCEL (LangChain Expression Language)** é a forma moderna de compor pipelines no LangChain. A ideia central é encadear componentes usando o operador pipe `|`.

```
ComponenteA | ComponenteB | ComponenteC
```

Cada componente recebe a saída do anterior como entrada. Os três componentes mais comuns são:

- **`ChatPromptTemplate`:** formata o texto de entrada em um prompt estruturado para o LLM.
- **`ChatGroq` (o LLM):** recebe o prompt e gera uma resposta.
- **`StrOutputParser`:** extrai apenas o texto da resposta do LLM, descartando metadados.

In [ ]:
# ============================================================================
# 3. EXEMPLO 1: CHAIN SIMPLES COM LCEL
# ============================================================================

from langchain_core.prompts import ChatPromptTemplate   # Construtor de prompts
from langchain_groq import ChatGroq                     # Modelo de linguagem via Groq
from langchain_core.output_parsers import StrOutputParser  # Extrai texto puro da resposta

print("\n" + "="*60)
print("EXEMPLO 1: Chain Simples (Prompt → LLM → Parser)")
print("="*60)

# Criar o LLM: define qual modelo será usado e quão determinístico ele é
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.1)

# Criar chain com LCEL usando o operador | (pipe)
# O fluxo é: template de prompt → LLM → extrator de texto
chain_simples = (
    ChatPromptTemplate.from_template("Explique {topico} em uma frase.")  # {topico} é uma variável substituída no invoke()
    | llm                                                                  # Envia o prompt ao modelo
    | StrOutputParser()                                                    # Transforma o objeto AIMessage em string pura
)

# Executar a chain passando um dicionário com os valores das variáveis do prompt
resultado = chain_simples.invoke({"topico": "machine learning"})
print(f"\nInput: machine learning")
print(f"Output:\n{resultado}")


EXEMPLO 1: Chain Simples (Prompt → LLM → Parser)

Input: machine learning
Output:
Machine learning é um campo da inteligência artificial que permite que computadores aprendam padrões e tomem decisões a partir de dados, sem serem explicitamente programados para cada tarefa.


---
## Seção 3: Pipelines com Pré e Pós-processamento

No mundo real, os dados de entrada raramente chegam limpos. Usuários digitam em maiúsculas, adicionam espaços extras ou formatos inesperados. Da mesma forma, a saída do LLM pode precisar ser formatada antes de chegar ao usuário final.

### O que é `RunnableLambda`?

`RunnableLambda` é um wrapper que transforma **qualquer função Python comum** em um componente compatível com o operador `|` do LCEL. Isso permite inserir lógica arbitrária em qualquer ponto do pipeline.

```python
meu_runnable = RunnableLambda(minha_funcao)
```

### Diagrama do pipeline deste exemplo

```
Entrada bruta → [pre_processamento] → [prompt] → [llm] → [StrOutputParser] → [pos_processamento] → Saída formatada
```

In [ ]:
# ============================================================================
# 4. EXEMPLO 2: PIPELINE COM PRÉ E PÓS-PROCESSAMENTO
# ============================================================================

from langchain_core.runnables import RunnableLambda  # Transforma funções Python em componentes LCEL

print("\n" + "="*60)
print("EXEMPLO 2: Pipeline com Pré-processamento")
print("="*60)

# Etapa 1: Pré-processamento
def pre_processar(entrada):
    """Limpa e valida a entrada."""
    # Remove espaços nas bordas e converte para minúsculas para normalizar a entrada
    texto = entrada.get("pergunta", "").strip().lower()
    return {"pergunta": texto}  # Retorna dicionário para a próxima etapa receber pelo mesmo formato

# Envolver a função em RunnableLambda para integrá-la ao pipeline LCEL
pre_processo = RunnableLambda(
    pre_processar,
    name="pre_processamento"  # Nome opcional, útil para debugging e visualização do pipeline
)

# Etapa 2: Prompt + LLM (reutiliza o llm criado na seção anterior)
prompt = ChatPromptTemplate.from_template(
    "Responda de forma técnica: {pergunta}"
)

# Etapa 3: Pós-processamento
def pos_processar(texto):
    """Formata a saída."""
    # Adiciona um cabeçalho padronizado à resposta para identificação visual
    return f"[RESPOSTA]\n{texto.strip()}"

pos_processo = RunnableLambda(
    pos_processar,
    name="pos_processamento"
)

# Montar pipeline: cada | passa a saída de um componente como entrada do próximo
pipeline = pre_processo | prompt | llm | StrOutputParser() | pos_processo

# Executar: note que a entrada tem espaços e está em maiúsculas — o pré-processamento vai limpar
resultado = pipeline.invoke({"pergunta": "  O QUE É RECURSÃO?  "})
print(f"\nInput: '  O QUE É RECURSÃO?  '")
print(f"Output:\n{resultado}")



EXEMPLO 2: Pipeline com Pré-processamento

Input: '  O QUE É RECURSÃO?  '
Output:
[RESPOSTA]
**Recursão – definição técnica**

Recursão é um método de definição (ou construção) de objetos, funções ou algoritmos em que o próprio objeto ou função é referenciado dentro da sua própria definição. Em ciência da computação e matemática, uma **função recursiva** é aquela que, ao ser invocada, pode chamar a si mesma com argumentos “menores” ou “mais simples”, de modo que, após um número finito de chamadas, chega‑se a um caso trivial conhecido como **caso base** (ou caso de parada).  

Formalmente, uma função \(f\) é recursiva se existir:

1. **Caso base** \(B\): uma condição que não envolve chamada recursiva e que devolve um resultado conhecido.  
2. **Caso recursivo** \(R\): uma expressão que invoca \(f\) com argumentos “aproximados” ao caso base.

\[
f(x) = 
\begin{cases}
B(x) & \text{se } P_{\text{base}}(x) \\
R\bigl(x, f(g_1(x)), f(g_2(x)),\dots\bigr) & \text{se } \neg P_{\text{base}}(x)
\

---
## Seção 4: Tools Customizadas

Uma das capacidades mais poderosas dos LLMs modernos é o **uso de ferramentas (tool calling)**. Em vez de apenas gerar texto, o modelo pode decidir invocar funções externas para buscar dados reais, como preços, clima ou informações de um banco de dados.

### Como funciona o `@tool`?

O decorator `@tool` transforma uma função Python comum em uma ferramenta que o LLM conhece. O LangChain extrai automaticamente:

- O **nome** da função como identificador da tool.
- A **docstring** como descrição: o LLM lê essa descrição para decidir quando e como usar a ferramenta.
- As **type annotations** dos parâmetros para entender o que deve ser passado.

### O que é `bind_tools`?

`bind_tools` associa uma lista de tools a um LLM. A partir daí, quando o modelo receber uma pergunta relevante, ele pode responder com uma **requisição de uso de tool** em vez de texto direto.

In [ ]:
# ============================================================================
# 5. EXEMPLO 3: TOOL CUSTOMIZADA
# ============================================================================

from langchain_core.tools import tool  # Decorator que transforma funções em tools do LangChain

print("\n" + "="*60)
print("EXEMPLO 3: Tool Customizada")
print("="*60)

@tool  # Este decorator registra a função como uma tool que o LLM pode usar
def consultar_preco(produto: str) -> str:
    """Consulta o preço de um produto.
    Use esta tool quando o usuário perguntar sobre preço."""
    # Dicionário simulando um banco de dados de preços
    precos = {
        "notebook": "R$ 3.500",
        "mouse": "R$ 150",
        "teclado": "R$ 450",
        "monitor": "R$ 1.200"
    }
    # Busca o preço pelo nome do produto (em minúsculas para evitar erros de capitalização)
    return precos.get(produto.lower(), "Produto não encontrado")

# Bind a tool ao LLM: o modelo agora sabe que essa tool existe e como usá-la
llm_com_tool = llm.bind_tools([consultar_preco])

# Simular uso
print("\nLLM recebe pergunta sobre preço:")
pergunta = "Qual o preço do notebook?"
# O LLM pode responder com uma chamada de tool em vez de texto
resposta = llm_com_tool.invoke(pergunta)

print(f"Pergunta: {pergunta}")
# tool_calls é preenchido quando o LLM decide usar uma tool
print(f"Tool chamada: {resposta.tool_calls if hasattr(resposta, 'tool_calls') else 'Nenhuma'}")

# Executar a tool manualmente para demonstrar
if hasattr(resposta, 'tool_calls') and resposta.tool_calls:
    tool_call = resposta.tool_calls[0]                        # Pega a primeira chamada de tool
    resultado_tool = consultar_preco.invoke(tool_call['args'])  # Executa a tool com os argumentos sugeridos pelo LLM
    print(f"Resultado: {resultado_tool}")
else:
    # Se o LLM não chamou a tool, chamar diretamente
    resultado_tool = consultar_preco.invoke("notebook")
    print(f"Resultado da tool: {resultado_tool}")



EXEMPLO 3: Tool Customizada

LLM recebe pergunta sobre preço:
Pergunta: Qual o preço do notebook?
Tool chamada: [{'name': 'consultar_preco', 'args': {'produto': 'notebook'}, 'id': 'fc_1d34e800-91ab-4d06-9a40-0baba2eaec90', 'type': 'tool_call'}]
Resultado: R$ 3.500


---
## Seção 5: Roteamento Dinâmico com RunnableBranch

Em muitas aplicações reais, perguntas diferentes exigem tratamentos diferentes. Um assistente de suporte técnico não deve responder de forma técnica para perguntas cotidianas, e vice-versa.

### O que é `RunnableBranch`?

`RunnableBranch` é o equivalente a um `if/elif/else` dentro de um pipeline LCEL. Ele recebe pares `(condição, chain)` e executa a primeira chain cuja condição for verdadeira. Se nenhuma condição for verdadeira, executa a chain padrão (o último argumento, sem condição).

```python
router = RunnableBranch(
    (condicao_1, chain_A),   # Se condicao_1 → executa chain_A
    (condicao_2, chain_B),   # Se condicao_2 → executa chain_B
    chain_padrao             # Senão → executa chain_padrao
)
```

Cada condição é um `RunnableLambda` que retorna `True` ou `False`.

In [ ]:
# ============================================================================
# 6. EXEMPLO 4: ROTEAMENTO COM RUNNABLEBRANCH
# ============================================================================

from langchain_core.runnables import RunnableBranch  # Componente de roteamento condicional

print("\n" + "="*60)
print("EXEMPLO 4: Roteamento (RouterChain moderno)")
print("="*60)

# Chain para perguntas técnicas: usa prompt mais formal e preciso
chain_tecnica = (
    ChatPromptTemplate.from_template(
        "Responda tecnicamente e com precisão: {pergunta}"
    )
    | llm
    | StrOutputParser()
)

# Chain para perguntas simples: usa prompt mais acessível e direto
chain_simples_rota = (
    ChatPromptTemplate.from_template(
        "Responda de forma acessível: {pergunta}"
    )
    | llm
    | StrOutputParser()
)

# Função de classificação: retorna True se a pergunta contém palavras técnicas
def eh_tecnica(entrada):
    """Detecta se pergunta é técnica."""
    palavras_tecnicas = ["api", "código", "bug", "algoritmo", "database"]
    # any() retorna True se ao menos uma palavra técnica estiver na pergunta
    return any(p in entrada["pergunta"].lower() for p in palavras_tecnicas)

# Criar router com RunnableBranch
# Estrutura: (condição_como_runnable, chain_a_executar), ..., chain_padrão
router = RunnableBranch(
    (RunnableLambda(eh_tecnica), chain_tecnica),  # Rota técnica
    chain_simples_rota                             # Rota padrão (perguntas simples)
)

# Testar
print("\nTeste 1 - Pergunta técnica:")
resultado1 = router.invoke({"pergunta": "Como funciona uma API REST?"})
print(f"Resposta:\n   {resultado1[:100]}...")

print("\nTeste 2 - Pergunta simples:")
resultado2 = router.invoke({"pergunta": "Qual é a capital da França?"})
print(f"Resposta:\n   {resultado2[:100]}...")


EXEMPLO 4: Roteamento (RouterChain moderno)

Teste 1 - Pergunta técnica:
Resposta:
   **API REST (Representational State Transfer)**  

Uma API REST é um conjunto de recursos expostos po...

Teste 2 - Pergunta simples:
Resposta:
   A capital da França é **Paris**....


---
### Milestone 2: Roteamento Funcionando

**O que se espera?**  
A pergunta sobre API deve ser roteada para `chain_tecnica` (resposta mais formal e precisa). A pergunta sobre a capital da França deve ir para `chain_simples_rota` (resposta mais direta e acessível).

**Sua tarefa:**  
- Execute a célula e compare os dois estilos de resposta. Você consegue identificar diferenças no tom?
- Adicione a palavra `"framework"` à lista `palavras_tecnicas` e teste com a pergunta `"O que é um framework?"`. O roteamento muda?
- Crie um terceiro teste com a pergunta `"O que é um bug de software?"` e verifique qual rota é acionada, dado que `"bug"` já está na lista.

---

---
## Seção 6: Resiliência com Retry e Fallback

Aplicações de produção precisam lidar com falhas. LLMs externos podem ficar indisponíveis, retornar erros de rate limit ou produzir respostas inválidas. O LangChain oferece dois mecanismos nativos para aumentar a robustez de pipelines:

### Retry

`.with_retry(stop_after_attempt=N)` configura o componente para tentar novamente automaticamente em caso de falha. É útil para erros transitórios de rede ou rate limit.

### Fallback

`.with_fallbacks([outro_llm])` define um modelo alternativo que será usado caso o modelo principal falhe em todas as tentativas. O mecanismo tenta o modelo principal primeiro. Se ele falhar, passa para o próximo da lista.

### Diagrama de resiliência

```
Requisição → llm_principal → [Falha?] → llm_fallback → Resposta
```

> **Atenção:** neste notebook o modelo principal e o fallback estão definidos com nomes diferentes, mas o funcionamento é idêntico. Em produção, você usaria modelos de capacidades ou custos diferentes.

In [ ]:
# ============================================================================
# 7. EXEMPLO 5: TRATAMENTO DE ERROS COM RETRY E FALLBACK
# ============================================================================

from langchain_groq import ChatGroq  # Reimportado para deixar o exemplo autocontido

print("\n" + "="*60)
print("EXEMPLO 5: Retry e Fallback")
print("="*60)

# LLM com retry nativo: tenta até 3 vezes antes de lançar exceção
llm_com_retry = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.7
).with_retry(stop_after_attempt=3)  # Configura até 3 tentativas automáticas em caso de falha

# Fallback para modelo diferente se falhar
llm_principal = ChatGroq(model="openai/gpt-oss-120b", temperature=0)   # Modelo preferencial
llm_fallback = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)  # Modelo de reserva

# Combina os dois: tenta o principal e, se falhar, usa o fallback automaticamente
llm_robusto = llm_principal.with_fallbacks([llm_fallback])

# Chain robusta: qualquer falha no llm_principal será tratada pelo mecanismo de fallback
chain_robusta = (
    ChatPromptTemplate.from_template("Resuma em 2 frases: {texto}")
    | llm_robusto                 # Usa o LLM com fallback configurado
    | StrOutputParser()
)

print("\nChain configurada com:")
print("   - Retry automático (até 3 tentativas)")
print("   - Fallback para modelo alternativo se falhar")

resultado = chain_robusta.invoke({
    "texto": "LangChain é um framework para construir aplicações com LLMs."
})
print(f"\nResultado:\n{resultado}")



EXEMPLO 5: Retry e Fallback

Chain configurada com:
   - Retry automático (até 3 tentativas)
   - Fallback para modelo alternativo se falhar

Resultado:
LangChain é um framework que simplifica a criação de aplicações que utilizam modelos de linguagem de grande escala (LLMs), oferecendo componentes reutilizáveis para gerenciamento de prompts, memória, agentes e integração com fontes de dados externas. Ele permite que desenvolvedores conectem LLMs a fluxos de trabalho complexos, facilitando a construção de chatbots, assistentes virtuais e outras soluções de IA de forma modular e escalável.


---
### Milestone 3: Pipeline Resiliente

**O que se espera?**  
A chain deve executar normalmente usando o `llm_principal`. A mensagem de configuração deve listar retry e fallback, e o resultado deve ser um resumo de duas frases sobre o LangChain.

**Sua tarefa:**  
- Altere o texto enviado para um tema diferente, como uma notícia ou conceito técnico de sua escolha.
- Reflita: em que cenário de produção faria sentido ter um `llm_fallback` com um modelo diferente e mais barato? Escreva sua resposta em uma nova célula Markdown abaixo desta.

---

---
## Seção 7: Desafio Prático Integrador

### O que você vai construir?

Um pipeline que:

1. Recebe uma pergunta do usuário.
2. Classifica se é técnica ou não (usando sua própria lógica de classificação).
3. Roteia para a chain adequada com `RunnableBranch`.
4. Aplica pós-processamento na saída com `RunnableLambda`.

### Dicas antes de começar

- Reutilize os componentes já definidos nas seções anteriores (`chain_tecnica`, `chain_simples_rota`, etc.).
- Lembre-se de que o `RunnableBranch` espera que a condição seja um `RunnableLambda` que retorna booleano.
- O pós-processamento recebe uma `str` e deve retornar uma `str`.

Tente com diferentes perguntas para avaliar o funcionamento.

In [ ]:
# ============================================================================
# 8. DESAFIO PRÁTICO PARA OS ALUNOS
# ============================================================================

print("\n" + "="*60)
print("DESAFIO PRÁTICO")
print("="*60)
print("""
Construa um pipeline completo que:

1. Receba uma pergunta do usuário
2. Classifique se é técnica ou não
3. Se técnica → use chain_tecnica
   Se simples → use chain_simples
4. Aplique pós-processamento formatando a resposta

Dicas:
- Use RunnableBranch para roteamento
- Use RunnableLambda para pós-processamento
- Teste com diferentes perguntas

Comece aqui:
""")

# Template para resolver o desafio
desafio_template = '''
# Seu código aqui:

def eh_tecnica_custom(entrada):
    # TODO: implementar classificação
    pass

chain_desafio = (
    # TODO: montar o pipeline com roteamento
)

# Testar
resultado_desafio = chain_desafio.invoke({"pergunta": "sua pergunta aqui"})
print(resultado_desafio)
'''

print(desafio_template)

# ============================================================================
# 9. LINKS E REFERÊNCIAS
# ============================================================================

print("\n" + "="*60)
print("RECURSOS E DOCUMENTAÇÃO")
print("="*60)
print("""
Documentação oficial:
   - LangChain: https://python.langchain.com/docs/
   - Groq: https://groq.com/""");


DESAFIO PRÁTICO

Construa um pipeline completo que:

1. Receba uma pergunta do usuário
2. Classifique se é técnica ou não
3. Se técnica → use chain_tecnica
   Se simples → use chain_simples
4. Aplique pós-processamento formatando a resposta

Dicas:
- Use RunnableBranch para roteamento
- Use RunnableLambda para pós-processamento
- Teste com diferentes perguntas

Comece aqui:


# Seu código aqui:

def eh_tecnica_custom(entrada):
    # TODO: implementar classificação
    pass

chain_desafio = (
    # TODO: montar o pipeline com roteamento
)

# Testar
resultado_desafio = chain_desafio.invoke({"pergunta": "sua pergunta aqui"})
print(resultado_desafio)


RECURSOS E DOCUMENTAÇÃO

Documentação oficial:
   - LangChain: https://python.langchain.com/docs/
   - Groq: https://groq.com/


---
## Seção 8: Recursos e Próximos Passos

### Conceitos vistos neste notebook

| Conceito | Componente LangChain | Para que serve |
|---|---|---|
| Chain simples | `ChatPromptTemplate` + `ChatGroq` + `StrOutputParser` | Fluxo básico de prompt para texto. |
| Funções no pipeline | `RunnableLambda` | Inserir qualquer lógica Python no LCEL. |
| Ferramentas externas | `@tool` + `bind_tools` | Permitir que o LLM chame funções com dados reais. |
| Roteamento condicional | `RunnableBranch` | Direcionar o fluxo com base no conteúdo da entrada. |
| Resiliência | `.with_retry()` + `.with_fallbacks()` | Lidar com falhas de forma automática. |

### Onde estudar mais

Documentação oficial:

    - LangChain: https://python.langchain.com/docs/
    - Groq: https://groq.com/